# 00_env_config

Environment bootstrap for FabricOps Starter Kit notebooks.
This notebook defines environment-wide values and assembles framework config.
Reusable functions come from `fabricops_kit` package modules.


In [ ]:
# Import key functions needed by the environment bootstrap.
# Make sure the Fabric environment already has fabricops installed as a custom library.

from fabricops_kit import setup_metadata_tables
from fabricops_kit.fabric_input_output import (
    FabricStore,
    read_lakehouse_csv,
    read_lakehouse_table,
    write_lakehouse_table,
    read_warehouse_table,
    write_warehouse_table,
)
from fabricops_kit.config import (
    AIPromptConfig,
    FrameworkConfig,
    GovernanceConfig,
    LineageConfig,
    NotebookRuntimeConfig,
    PathConfig,
    QualityConfig,
    ReviewWorkflowConfig,
    DataAgreementConfig,
    setup_notebook,
)

In [ ]:
print("FabricOps Starter Kit environment configuration loaded.")


# List of configs

### Runtime Config

In [ ]:
# Change this if you want other prefixes for setup_notebook naming checks.
NOTEBOOK_PREFIXES = ("00_env_config", "01_agreement", "02_pipeline", "03_governance", "99_explore")

RUNTIME_CONFIG = NotebookRuntimeConfig(NOTEBOOK_PREFIXES)

### Path Config

In [ ]:
# Change this if needed for your own custom-defined environments, for example: dev, qat, prd.
ENV = "dev"
ENV_NAME = ENV

# Use "warn" during setup. Use "strict" when you want missing prerequisites to fail the bootstrap.
VALIDATION_MODE = "warn"

REQUIRED_TARGETS = ["source", "unified", "product", "metadata"]

# Change these placeholders to your actual Fabric workspace and item IDs.
# You can get workspace_id and item_id from the Fabric URL.
# Target names should align with REQUIRED_TARGETS above.
ENV_PATHS = {
    ENV: {
        "source": FabricStore(
            env=ENV,
            workspace_id="00000000-0000-0000-0000-000000000001",
            item_id="11111111-1111-1111-1111-111111111111",
            name="lh_source_dev",
            kind="lakehouse",
        ),
        "unified": FabricStore(
            env=ENV,
            workspace_id="00000000-0000-0000-0000-000000000001",
            item_id="22222222-2222-2222-2222-222222222222",
            name="lh_unified_dev",
            kind="lakehouse",
        ),
        "product": FabricStore(
            env=ENV,
            workspace_id="00000000-0000-0000-0000-000000000001",
            item_id="33333333-3333-3333-3333-333333333333",
            name="wh_product_dev",
            kind="warehouse",
        ),
        "metadata": FabricStore(
            env=ENV,
            workspace_id="00000000-0000-0000-0000-000000000001",
            item_id="44444444-4444-4444-4444-444444444444",
            name="lh_metadata_dev",
            kind="lakehouse",
        ),
    }
}

PATH_CONFIG = PathConfig(paths=ENV_PATHS)

### 01_agreement Metadata Intake Config

The two `01_agreement` widgets expose only lightweight business fields. Add organization-specific fields here; the widgets store those values in `custom_fields_json` without changing package code or table schemas.


In [ ]:
DATA_AGREEMENT_CONFIG = DataAgreementConfig(
    metadata_tables={
        "data_steward": "METADATA_DATA_STEWARD",
        "data_agreement": "METADATA_DATA_AGREEMENT",
        "data_agreement_evidence": "METADATA_DATA_AGREEMENT_EVIDENCE",
    },
    steward_role_options=[
        "Data Owner",
        "Data Steward",
        "Data Custodian",
        "Governance Reviewer",
        "Business Approver",
    ],
    data_steward_widget={
        "visible_columns": [
            "steward_name", "steward_role", "contact", "effective_from", "effective_to",
        ],
        "custom_fields": [
            {
                "key": "optional",
                "label": "Optional",
                "type": "text",
                "required": False,
                "help": "Optional group of users or organization unit covered by the agreement.",
            },
            {
                "key": "optional_dropdown",
                "label": "Optional dropdown",
                "type": "select",
                "required": False,
                "options": ["ODI", "Faculty", "Department", "Research Group", "Other"],
            },
        ],
    },
    data_agreement_widget={
        "visible_columns": [
            "agreement_name", "domain", "steward_id", "recipient", "start_date",
            "expiry_date", "business_purpose", "approved_usage_internal",
            "approved_usage_external", "approved_usage_research",
        ],
        "custom_fields": [
            {
                "key": "optional",
                "label": "Optional",
                "type": "text",
                "required": False,
                "help": "Optional group of users or organization unit covered by the agreement.",
            },
            {
                "key": "optional_dropdown",
                "label": "Optional dropdown",
                "type": "select",
                "required": False,
                "options": ["ODI", "Faculty", "Department", "Research Group", "Other"],
            },
        ],
    },
)


Creates or checks agreement, registry, catalogue, lineage, business-context, DQ-rule, and classification metadata tables in `CONFIG.path_config.paths[ENV]["metadata"]`.

`01_agreement` renders a tabbed intake app with steward maintenance, agreement maintenance, and optional evidence upload. Runtime audit fields are added automatically and remain hidden from normal widget users.


### Other config

In [ ]:

QUALITY_CONFIG = QualityConfig()
GOVERNANCE_CONFIG = GovernanceConfig()
REVIEW_WORKFLOW_CONFIG = ReviewWorkflowConfig()
LINEAGE_CONFIG = LineageConfig()


### AI Prompts Config

###

In [ ]:
BUSINESS_CONTEXT_PROMPT_TEMPLATE = """
You are helping draft the business context for a FabricOps data sharing agreement notebook.

This output is advisory only.
A human data owner, steward, analyst, or engineer must review and approve it before it is treated as official agreement context.

Your job is to convert agreement metadata, source descriptions, notebook notes, and available evidence into clear business context that downstream optional exploration and delivery pipeline notebooks can use.

Focus on:
1. What business question or operational need this data supports.
2. Who owns or stewards the data.
3. Who is allowed to use the data.
4. What the data can be used for.
5. What the data must not be used for.
6. Any sensitivity, privacy, or governance concerns.
7. Any known limitations, caveats, or open questions.

Do not invent facts.
If information is missing, mark it as an open question.
Do not approve access, usage, classifications, or data quality rules.
Do not write legal language.
Use plain business language.

Return only a Python dictionary named BUSINESS_CONTEXT using this shape:

BUSINESS_CONTEXT = {
    "agreement_id": "{agreement_id}",
    "agreement_name": "{agreement_name}",
    "business_purpose": "Plain explanation of why this data is needed.",
    "primary_users": ["team_or_role"],
    "data_owner": "Known owner or 'unknown'.",
    "data_steward": "Known steward or 'unknown'.",
    "approved_usage": ["usage"],
    "restricted_usage": ["restriction"],
    "key_business_terms": {
        "term": "definition"
    },
    "known_limitations": ["limitation"],
    "governance_notes": ["note"],
    "open_questions": ["question"]
}

Agreement metadata:
{agreement_metadata}

Source evidence:
{source_evidence}

User notes:
{user_notes}
""".strip()

In [ ]:
DQ_RULE_SUGGESTION_PROMPT_TEMPLATE = """
You are helping draft candidate FabricOps-native data quality rules for a Microsoft Fabric pipeline.

These suggestions are advisory drafts only. A human reviewer must approve, edit, or reject every rule before enforcement.

Suggest FabricOps-native DQ rules only. FabricOps supports 23 FabricOps-native DQ rule types. Use only supported rule_type values. Do not invent rule types. Prefer simple named rules before expression_true. Treat expression_true as the Custom expression rule and use it only when no smaller named rule can express the requirement; only trusted reviewers should approve expression rules.

Rule selection principles:
- Suggest DQ rules only when the column profile or business context gives enough evidence.
- Prefer the smallest named rule that expresses the requirement.
- Use column profile evidence: data_type, row_count, null_count, null_percent, distinct_count, distinct_percent, min_value, max_value, and observed_values_sample.
- Do not suggest datatype/schema rules; schema validation is separate.
- Do not suggest source stability rules; stability checks are separate.
- Schema guardrails and source stability are separate FabricOps layers.
- Do not suggest row filtering or quarantine behavior; FabricOps v1 reports/tags outcomes and error severity blocks unsafe downstream writes.
- Use expression_true only as the Custom expression escape hatch when no smaller rule fits.

Data type / constraint-shape guidance and required parameters for all 23 rule types:

Rule catalogue guidance:
Completeness:
- not_null: any profiled column where every row must have a non-null value. Required: columns.
- null_rate_below: any profiled column where some nulls are allowed but the null percentage must stay below a threshold. Required: columns, max_null_percent.
- non_empty_string: string columns where blank or whitespace-only values should fail. Required: columns.
- required_when: any target column required only when a condition is true. Required: columns, condition.

Uniqueness:
- unique: one column should uniquely identify rows. Required: columns.
- unique_combination: a composite business key or table grain should be unique. Required: columns.

Allowed / blocked values:
- accepted_values: governed allowed values for string, numeric, boolean, or code columns. Required: columns, allowed_values.
- not_in_values: placeholder, blocked, retired, or invalid values must not appear. Required: columns, blocked_values.

Numeric / comparable ranges:
- between: comparable value must be within min and/or max. Common for numeric, date, timestamp, or consistently formatted strings. Required: columns, at least one of min_value or max_value.
- greater_than: value must be greater than threshold. Required: columns, value.
- greater_than_or_equal: value must be greater than or equal to threshold. Required: columns, value.
- less_than: value must be less than threshold. Required: columns, value.
- less_than_or_equal: value must be less than or equal to threshold. Required: columns, value.

Pattern and date rules:
- regex_match: string column must match a pattern. Required: columns, regex_pattern.
- date_not_future: date or timestamp column must not be in the future. Required: columns.
- date_between: date or timestamp column must be within a date range. Required: columns, min_value, max_value.
- freshness: date or timestamp column must be recent enough. Required: columns, max_age_days.
- max_age_days: date or timestamp value must not be older than max age. Required: columns, max_age_days.

Cross-column logic:
- column_pair_equal: two compatible columns must be equal. Required: exactly two columns.
- column_a_gte_column_b: first comparable column must be greater than or equal to second. Required: exactly two columns.
- column_a_gt_column_b: first comparable column must be greater than second. Required: exactly two columns.
- value_when: target column must equal expected_value when condition is true. Required: one columns value, condition, expected_value.

Advanced:
- expression_true: Custom expression rule. Use only when no named rule can express the business requirement. Required: expression. Prefer not to use it unless there is clear business logic evidence.

Priority guide:
- If checking presence, choose not_null, null_rate_below, non_empty_string, or required_when.
- If checking duplicate grain, choose unique or unique_combination.
- If checking governed categories, choose accepted_values.
- If checking invalid placeholders, choose not_in_values.
- If checking numeric or date bounds, choose between or threshold rules.
- If checking formatted text, choose regex_match.
- If checking recency, choose freshness or max_age_days.
- If checking relationship between columns, choose cross-column rules.
- If checking conditional business logic, choose required_when or value_when.
- Use expression_true only after all named rules are insufficient.

Evidence guidance:
- Use not_null only when business context indicates the column is mandatory or null_count is already zero and the column looks required.
- Use accepted_values only when observed_values_sample or business context shows a stable controlled set.
- Use unique when distinct_count equals or is expected to equal row_count.
- Use unique_combination only when business context indicates a composite key/table grain.
- Use null_rate_below when nulls are expected but should stay within tolerance.
- Use warning severity by default unless the rule protects a key, required business field, financial measure, compliance field, or downstream join/grain integrity.
- Use error severity when bad data would make output unsafe or misleading.

Output guardrails:
- Every suggestion must include rule_id, rule_type, columns, severity, description, and required parameters.
- rule_id must be lower snake case.
- columns must contain actual column names from the profile/context.
- Do not invent columns.
- Do not invent rule types.
- Do not output markdown.
- Do not output comments.
- Return valid JSON only.

What belongs outside DQ:
- Do not suggest schema rules such as required_columns, expected_schema, or datatype checks.
- Do not suggest source stability rules.
- Schema guardrails and source stability are separate FabricOps layers.

Return valid JSON only in this shape:
{"DQ_RULES":{"{table_name}":[{"rule_id":"lower_snake_case_rule_id","rule_type":"not_null","columns":["column_name"],"severity":"warning","description":"Plain business explanation."}]}}

Table name: {table_name}
Business context: {business_context}
Column profile row:
Column name: {column_name}
Data type: {data_type}
Row count: {row_count}
Null count: {null_count}
Null percent: {null_percent}
Distinct count: {distinct_count}
Distinct percent: {distinct_percent}
Minimum value: {min_value}
Maximum value: {max_value}
Observed values sample: {observed_values_sample}
""".strip()


In [ ]:
GOVERNANCE_PERSONAL_IDENTIFIER_PROMPT_TEMPLATE = """
You are helping identify potential personal identifiers in a FabricOps data sharing agreement or optional exploration notebook.

This output is advisory only.
A human data steward or governance reviewer must approve the final classification.

Your job is to review column names, descriptions, data types, samples, and business context to identify columns that may directly or indirectly identify a person.

Classify each candidate as one of:

1. direct_identifier
   Use when the column can directly identify a person.
   Examples: staff_id, student_id, nric, passport_number, email, phone_number, full_name.

2. quasi_identifier
   Use when the column may identify a person when combined with other fields.
   Examples: date_of_birth, postal_code, department, nationality, gender, job_title, programme, cohort.

3. sensitive_personal_attribute
   Use when the column describes sensitive personal information.
   Examples: disability, health, religion, ethnicity, disciplinary status, financial aid, salary, performance rating.

4. not_personal_identifier
   Use when the column does not appear to identify a person.

Rules:
- Do not over-classify generic operational fields unless there is a clear person-related meaning.
- If uncertain, mark confidence as "medium" or "low" and explain why.
- Do not decide final access policy.
- Do not recommend masking unless the evidence supports it.
- Use plain governance language.

Return only a Python dictionary named PERSONAL_IDENTIFIER_CANDIDATES using this shape:

PERSONAL_IDENTIFIER_CANDIDATES = {
    "{table_name}": [
        {
            "column": "column_name",
            "classification": "direct_identifier | quasi_identifier | sensitive_personal_attribute | not_personal_identifier",
            "confidence": "high | medium | low",
            "reason": "Plain explanation.",
            "recommended_review_action": "approve | review | ignore"
        }
    ]
}

Table name:
{table_name}

Business context:
{business_context}

Column evidence:
Column name: {column_name}
Data type: {data_type}
Description: {column_description}
Observed values sample: {observed_values_sample}
""".strip()

In [ ]:
GOVERNANCE_CANDIDATE_PROMPT_TEMPLATE = """
You are helping draft candidate governance classifications for a FabricOps data sharing agreement or pipeline contract.

These classifications are advisory only.
A human data steward, data owner, or governance reviewer must approve them before they are used for enforcement, access review, masking, or publication.

Use the available business context, column profile, personal identifier candidates, and agreement restrictions to suggest governance metadata.

Use these sensitivity levels:

1. public
   Data can be shared publicly without meaningful risk.

2. internal
   Data is intended for internal operational use but is not sensitive.

3. confidential
   Data has business, operational, contractual, or personal sensitivity and should be access-controlled.

4. restricted
   Data contains highly sensitive personal, regulated, security, or privileged information and requires strict controls.

Use these privacy labels:

1. no_personal_data
2. personal_data
3. sensitive_personal_data

Rules:
- Do not invent policy.
- Do not approve access.
- Do not assign "public" if any personal identifiers are present.
- Use "restricted" only when the column or dataset clearly requires strict control.
- Dataset-level classification should reflect the highest reasonable sensitivity across the evidence.
- Explain the reason clearly.
- Include open questions when evidence is insufficient.

Return only a Python dictionary named GOVERNANCE_CANDIDATES using this shape:

GOVERNANCE_CANDIDATES = {
    "dataset": {
        "table_name": "{table_name}",
        "candidate_sensitivity": "public | internal | confidential | restricted",
        "candidate_privacy_label": "no_personal_data | personal_data | sensitive_personal_data",
        "confidence": "high | medium | low",
        "reason": "Plain explanation.",
        "open_questions": ["question"]
    },
    "columns": [
        {
            "column": "column_name",
            "candidate_sensitivity": "public | internal | confidential | restricted",
            "candidate_privacy_label": "no_personal_data | personal_data | sensitive_personal_data",
            "confidence": "high | medium | low",
            "reason": "Plain explanation.",
            "recommended_controls": ["access_control | masking | aggregation | review_only"]
        }
    ]
}

Table name:
{table_name}

Business context:
{business_context}

Agreement restrictions:
{agreement_restrictions}

Personal identifier candidates:
{personal_identifier_candidates}

Column profile:
{column_profile}
""".strip()

In [ ]:
GOVERNANCE_REVIEW_PROMPT_TEMPLATE = """
You are helping prepare governance candidates for human review in a FabricOps workflow.

The goal is not to auto-approve governance decisions.
The goal is to make the review easier, clearer, and safer.

Review the proposed governance candidates and produce:
1. A concise review summary.
2. Items that look reasonable for approval.
3. Items that need human attention.
4. Items that appear inconsistent or unsupported.
5. Open questions for the data owner or steward.

Rules:
- Do not mark anything as finally approved.
- Do not weaken a classification without strong evidence.
- If there is uncertainty, route to human review.
- Highlight conflicts between business context, personal identifier findings, and candidate sensitivity.
- Use plain language suitable for governance reviewers.

Return only a Python dictionary named GOVERNANCE_REVIEW using this shape:

GOVERNANCE_REVIEW = {
    "table_name": "{table_name}",
    "review_summary": "Concise summary.",
    "recommended_for_approval": [
        {
            "scope": "dataset | column",
            "name": "dataset_or_column_name",
            "recommendation": "Plain recommendation.",
            "reason": "Reason."
        }
    ],
    "needs_human_review": [
        {
            "scope": "dataset | column",
            "name": "dataset_or_column_name",
            "issue": "What needs review.",
            "suggested_question": "Question to ask."
        }
    ],
    "inconsistencies": [
        {
            "scope": "dataset | column",
            "name": "dataset_or_column_name",
            "issue": "Conflict or unsupported claim."
        }
    ],
    "open_questions": ["question"]
}

Table name:
{table_name}

Business context:
{business_context}

Governance candidates:
{governance_candidates}

Personal identifier candidates:
{personal_identifier_candidates}

Agreement restrictions:
{agreement_restrictions}
""".strip()

In [ ]:
REVIEW_SUMMARY_PROMPT_TEMPLATE = """
You are helping summarize FabricOps governance review evidence for a data product.

Use only the provided evidence. Do not invent lineage, rules, approvals, or ownership.
If something is missing, list it under open questions or follow-up actions.

Return a concise dictionary named REVIEW_SUMMARY with purpose, approved usage, reviewed controls, lineage notes, known risks, open questions, and recommended next actions.
""".strip()


In [ ]:
AI_PROMPTS = AIPromptConfig(
    business_context_prompt_template=BUSINESS_CONTEXT_PROMPT_TEMPLATE,
    dq_rule_suggestion_prompt_template=DQ_RULE_SUGGESTION_PROMPT_TEMPLATE,
    governance_personal_identifier_prompt_template=GOVERNANCE_PERSONAL_IDENTIFIER_PROMPT_TEMPLATE,
    governance_candidate_prompt_template=GOVERNANCE_CANDIDATE_PROMPT_TEMPLATE,
    governance_review_prompt_template=GOVERNANCE_REVIEW_PROMPT_TEMPLATE,
)


## Config Compiler & Bootstrap

In [ ]:
CONFIG = FrameworkConfig(
    path_config=PATH_CONFIG,
    notebook_runtime_config=RUNTIME_CONFIG,
    ai_prompt_config=AI_PROMPTS,
    quality_config=QUALITY_CONFIG,
    governance_config=GOVERNANCE_CONFIG,
    review_workflow_config=REVIEW_WORKFLOW_CONFIG,
    data_agreement_config=DATA_AGREEMENT_CONFIG,
    lineage_config=LINEAGE_CONFIG,
)

RUN_CONTEXT = setup_notebook(
    config=CONFIG,
    env=ENV,
    required_targets=REQUIRED_TARGETS,
)



In [ ]:
METADATA_TABLE_SETUP = setup_metadata_tables(
    spark=spark,
    config=CONFIG,
    env=ENV,
    require_active_steward=False,
)
AGREEMENT_METADATA_SETUP = METADATA_TABLE_SETUP["data_agreement"]
print("Metadata table setup:", METADATA_TABLE_SETUP)


In [ ]:
print("FabricOps environment bootstrap ready")
print(f"- env: {ENV}")
print(f"- validation mode: {VALIDATION_MODE}")
print(f"- source target: {CONFIG.path_config.paths[ENV]['source'].name}")
print(f"- unified target: {CONFIG.path_config.paths[ENV]['unified'].name}")
print(f"- product target: {CONFIG.path_config.paths[ENV]['product'].name}")
print(f"- metadata target: {CONFIG.path_config.paths[ENV]['metadata'].name}")
print(f"- 01_agreement metadata tables created/checked: {AGREEMENT_METADATA_SETUP['tables']}")
print(f"- 01_agreement widget table names: {CONFIG.data_agreement_config.metadata_tables}")
print(f"- data steward custom fields: {CONFIG.data_agreement_config.data_steward_widget['custom_fields']}")
print(f"- data agreement custom fields: {CONFIG.data_agreement_config.data_agreement_widget['custom_fields']}")
print(f"- steward readiness: {AGREEMENT_METADATA_SETUP['status']}")
print(f"  - message: {AGREEMENT_METADATA_SETUP['message']}")
print(f"  - active steward count: {AGREEMENT_METADATA_SETUP['active_steward_count']}")

if VALIDATION_MODE == "strict" and AGREEMENT_METADATA_SETUP["status"] != "ready":
    raise RuntimeError(AGREEMENT_METADATA_SETUP["message"])
